# Demo notebook for Langchain Kinetica extension

In [ ]:
import os
from langchain_kinetica import KineticaChatLLM, KineticaSqlOutputParser

k_url = os.environ["KINETICA_URL"]
k_login = os.environ["KINETICA_LOGIN"]
k_passwd = os.environ["KINETICA_PASSWORD"]

kdbc = KineticaChatLLM._create_kdbc(host=k_url, login=k_login, password=k_passwd)
kinetica_llm = KineticaChatLLM(kdbc=kdbc)

In [ ]:
# test load messages
from langchain_core.prompts import ChatPromptTemplate

ctx_messages = kinetica_llm.load_messages_from_context('telecom.chad_test')
ctx_messages.append(("human", "{input}"))

prompt_template = ChatPromptTemplate.from_messages(ctx_messages)
messages = prompt_template.format_messages(input="What is the invoice date and price of transactions made by customer C274529?")

prompt_template.pretty_print()

In [ ]:
# create and execute the chain 

chain = prompt_template | kinetica_llm | KineticaSqlOutputParser(kdbc=kdbc)
response = chain.invoke({"input": "What is the invoice date and price of transactions made by customer C274529?"})
print(f"SQL: {response.sql}")
response.dataframe